# Intermediate 10 — Authorization Observability & Audit Analytics for Agents

## Scenario

A `claims-agent` performs hundreds of delegated tool operations per hour.

Security needs to answer:

```text
Why was claim:483 updated?
Who delegated the authority?
Which workload executed it?
Which policy version allowed it?
Did the PEP enforce the exact approved operation?
Was this behavior normal?
Can we prove the sequence six months later?
```

We build an observability and evidence pipeline rather than a pile of unrelated logs.


In [ ]:
from datetime import datetime, timedelta, timezone
import uuid, json, random, hashlib, hmac, copy
import pandas as pd
import networkx as nx

NOW=datetime.now(timezone.utc)
random.seed(7)


## 1 — Canonical authorization decision event

In [ ]:
def decision_event(agent="claims-agent", action="claim.read",
                   resource="claim:483", decision="allow",
                   reason="ALLOW_TASK_SCOPE", risk="medium",
                   policy_version="v19", trace_id=None):
    return {
      "schema_version":"1.0",
      "event_type":"authorization.decision",
      "decision_id":str(uuid.uuid4()),
      "timestamp":NOW.isoformat(),
      "trace_id":trace_id or uuid.uuid4().hex,
      "principal":{"id":"user:alice","type":"user","tenant":"acme"},
      "agent":{"id":agent,"version":"4.3.1","owner":"claims-platform"},
      "workload":{
        "spiffe_id":f"spiffe://corp.example/prod/agents/{agent}",
        "image_digest":"sha256:approved",
        "attested":True
      },
      "delegation":{
        "id":"del:483",
        "chain":["user:alice",f"agent:{agent}"],
        "depth":1,
        "expires_at":(NOW+timedelta(hours=1)).isoformat()
      },
      "task":{"id":"task:483","resource":"claim:483"},
      "action":{"name":action},
      "resource":{"id":resource,"type":resource.split(":")[0],"tenant":"acme"},
      "policy":{
        "engine":"opa",
        "policy_version":policy_version,
        "bundle_version":"bundle-2026-08-18.19",
        "determining_policies":["claims-task-scope"]
      },
      "assurance":{"human_aal":2,"workload_attested":True},
      "risk":{"level":risk,"score":25 if risk=="medium" else 80},
      "decision":{"outcome":decision,"reason_code":reason},
      "enforcement":{"pep":"agent-gateway","enforced":True},
      "latency_ms":random.randint(4,35)
    }

event=decision_event()
print(json.dumps(event,indent=2))


## 2 — Generate realistic telemetry

In [ ]:
actions=["claim.read","claim.update","claim.export","claim.delete"]
events=[]
for i in range(500):
    action=random.choices(actions,weights=[70,20,7,3])[0]
    deny=random.random() < (0.04 if action!="claim.delete" else 0.5)
    events.append(decision_event(
      action=action,
      decision="deny" if deny else "allow",
      reason="DENY_TASK_SCOPE" if deny else "ALLOW_TASK_SCOPE",
      risk="critical" if action in {"claim.export","claim.delete"} else "medium"
    ))
df=pd.json_normalize(events)
df.head()


## 3 — Decision volume and outcome

In [ ]:
df["decision.outcome"].value_counts()


## 4 — Structured reason-code analytics

In [ ]:
df.groupby(["decision.outcome","decision.reason_code"]).size().sort_values(ascending=False)


## 5 — PDP latency

In [ ]:
df["latency_ms"].describe(percentiles=[.5,.95,.99])


## 6 — Correlation IDs

In [ ]:
e=events[0]
print({
 "trace_id":e["trace_id"],
 "decision_id":e["decision_id"],
 "task_id":e["task"]["id"],
 "delegation_id":e["delegation"]["id"]
})


## 7 — Delegation provenance graph

In [ ]:
chain=event["delegation"]["chain"]
G=nx.DiGraph()
G.add_edges_from(zip(chain,chain[1:]))
print(list(G.edges()))
print("depth:",nx.shortest_path_length(G,chain[0],chain[-1]))


## 8 — Deny-spike simulation

In [ ]:
timeline=[]
for minute in range(60):
    base=5
    denies=base + (40 if 35<=minute<=39 else 0) + random.randint(0,4)
    timeline.append({"minute":minute,"denies":denies})
tdf=pd.DataFrame(timeline)
baseline=tdf[tdf.minute<30].denies.mean()
tdf["spike"]=tdf.denies > baseline*3
tdf[tdf.spike]


## 9 — Correlate spike with policy deployment

In [ ]:
deploy_minute=34
print(tdf[(tdf.minute>=deploy_minute)&tdf.spike])
print("Investigate policy/config deployment immediately before spike.")


## 10 — Suspicious allows

In [ ]:
suspicious=df[
 (df["decision.outcome"]=="allow") &
 (df["risk.level"]=="critical")
]
suspicious[["agent.id","action.name","resource.id","risk.score","policy.policy_version"]].head()


## 11 — New agent-tool pair

In [ ]:
historical_pairs={
 ("claims-agent","claim.read"),
 ("claims-agent","claim.update")
}
today_pairs=set(zip(df["agent.id"],df["action.name"]))
print("new pairs:",today_pairs-historical_pairs)


## 12 — Policy-version drift

In [ ]:
v20=[decision_event(policy_version="v20",
                    decision="allow",
                    action="claim.export",
                    risk="critical") for _ in range(30)]
mixed=pd.concat([df,pd.json_normalize(v20)],ignore_index=True)
mixed.groupby("policy.policy_version")["decision.outcome"].value_counts(normalize=True)


## 13 — Delegation-depth anomaly

In [ ]:
depths=[1,1,1,1,1,2,1,1,4]
baseline_max=2
print("anomalies:",[d for d in depths if d>baseline_max])


## 14 — PEP bypass detection

In [ ]:
tool_calls=pd.DataFrame([
 {"tool_call_id":"t1","decision_id":events[0]["decision_id"],"tool":"claims","operation":"read"},
 {"tool_call_id":"t2","decision_id":None,"tool":"payments","operation":"create"},
])
print(tool_calls[tool_calls.decision_id.isna()])


## 15 — Verify denied operations were not executed

In [ ]:
decisions=pd.DataFrame([
 {"decision_id":"x1","outcome":"deny"},
 {"decision_id":"x2","outcome":"allow"},
])
executions=pd.DataFrame([
 {"decision_id":"x1","executed":True},
 {"decision_id":"x2","executed":True},
])
joined=decisions.merge(executions,on="decision_id",how="left")
joined[(joined.outcome=="deny") & (joined.executed==True)]


## 16 — Decision/action parameter binding

In [ ]:
def transaction_digest(tool,operation,resource,critical_args):
    body=json.dumps({
      "tool":tool,"operation":operation,"resource":resource,
      "critical_args":critical_args
    },sort_keys=True,separators=(",",":"))
    return hashlib.sha256(body.encode()).hexdigest()

approved=transaction_digest("payments","create","account:42",{"amount":100,"currency":"CAD"})
changed=transaction_digest("payments","create","account:42",{"amount":10000,"currency":"CAD"})
print("same authorized transaction:",approved==changed)


## 17 — Sensitive-field classification

In [ ]:
CLASSIFICATION={
 "principal.id":"pseudonymous",
 "input.token":"secret",
 "tool.arguments.customer_ssn":"restricted",
 "resource.id":"confidential",
 "decision.reason_code":"internal"
}
CLASSIFICATION


## 18 — Redaction

In [ ]:
def redact(obj):
    x=copy.deepcopy(obj)
    if "token" in x:
        x["token"]="**REDACTED**"
    if x.get("tool",{}).get("arguments",{}).get("customer_ssn"):
        x["tool"]["arguments"]["customer_ssn"]="**REDACTED**"
    return x

sensitive={
 "token":"secret-token",
 "tool":{"arguments":{"customer_ssn":"123-45-6789","claim":"483"}}
}
print(redact(sensitive))


## 19 — HMAC pseudonymization

In [ ]:
ANALYTICS_KEY=b"training-key-not-for-production"

def pseudonym(value):
    digest=hmac.new(ANALYTICS_KEY,value.encode(),hashlib.sha256).hexdigest()
    return "usr:"+digest[:16]

print(pseudonym("user:alice"))
print(pseudonym("user:alice"))


## 20 — OPA-style masking exercise

In [ ]:
opa_event={
 "input":{
   "token":"secret",
   "user":{"email":"alice@example.com"},
   "tool":{"arguments":{"customer_ssn":"123-45-6789"}}
 }
}
print("Use policies/opa/observability.rego with a real OPA instance.")


## 21 — Cedar-style diagnostics

In [ ]:
cedar_allow={
 "decision":"Allow",
 "diagnostics":{
   "reason":["policy-claims-task"],
   "errors":[]
 }
}
cedar_default_deny={
 "decision":"Deny",
 "diagnostics":{
   "reason":[],
   "errors":[]
 }
}
cedar_error={
 "decision":"Deny",
 "diagnostics":{
   "reason":[],
   "errors":["policy-risk: missing attribute"]
 }
}
print(cedar_allow, cedar_default_deny, cedar_error, sep="\n")


## 22 — Normalize diagnostics

In [ ]:
def cedar_reason(r):
    d=r["diagnostics"]
    if d["errors"]:
        return "DENY_POLICY_EVALUATION_ERROR"
    if r["decision"]=="Deny" and not d["reason"]:
        return "DENY_POLICY_DEFAULT"
    return "ALLOW_POLICY_MATCH" if r["decision"]=="Allow" else "DENY_FORBID"

for x in [cedar_allow,cedar_default_deny,cedar_error]:
    print(cedar_reason(x))


## 23 — OpenTelemetry span model

In production, use the OpenTelemetry SDK rather than inventing a proprietary trace system.

A security span might carry low-risk structured attributes:

```text
authz.decision.id
authz.decision.outcome
authz.reason_code
agent.id
authz.policy.version
authz.risk.level
```

Be cautious with high-cardinality identifiers and never attach secrets.


In [ ]:
span_example={
 "name":"authorize claim.update",
 "trace_id":event["trace_id"],
 "attributes":{
   "authz.decision.id":event["decision_id"],
   "authz.decision.outcome":event["decision"]["outcome"],
   "authz.reason_code":event["decision"]["reason_code"],
   "agent.id":event["agent"]["id"],
   "authz.policy.version":event["policy"]["policy_version"],
   "authz.risk.level":event["risk"]["level"]
 }
}
span_example


## 24 — Metrics with controlled cardinality

In [ ]:
metrics={
 "decision_count":df.groupby(["decision.outcome","risk.level"]).size().to_dict(),
 "p95_latency_ms":float(df.latency_ms.quantile(.95)),
 "high_risk_allows":int(((df["decision.outcome"]=="allow")&(df["risk.level"]=="critical")).sum())
}
metrics


## 25 — SIEM-style detection rules

In [ ]:
DETECTIONS=[
 {"id":"AUTHZ-001","name":"Protected tool call without decision ID","severity":"critical"},
 {"id":"AUTHZ-002","name":"Critical allow from new agent-action pair","severity":"high"},
 {"id":"AUTHZ-003","name":"Deny spike after policy deployment","severity":"high"},
 {"id":"AUTHZ-004","name":"Quarantined agent allowed","severity":"critical"},
 {"id":"AUTHZ-005","name":"Delegation depth above approved maximum","severity":"high"},
]
pd.DataFrame(DETECTIONS)


## 26 — Hash-chained audit evidence

In [ ]:
def canonical(x):
    return json.dumps(x,sort_keys=True,separators=(",",":"),default=str)

def build_chain(records):
    previous="GENESIS"
    chain=[]
    for r in records:
        payload=canonical(r)
        digest=hashlib.sha256((previous+payload).encode()).hexdigest()
        chain.append({"record":r,"previous_hash":previous,"hash":digest})
        previous=digest
    return chain

audit_records=[
 {"decision_id":"a1","decision":"allow","policy":"v19"},
 {"decision_id":"a2","decision":"deny","policy":"v19"},
 {"decision_id":"a3","decision":"allow","policy":"v20"},
]
chain=build_chain(audit_records)
chain


## 27 — Verify chain integrity

In [ ]:
def verify_chain(chain):
    previous="GENESIS"
    for item in chain:
        expected=hashlib.sha256(
          (previous+canonical(item["record"])).encode()
        ).hexdigest()
        if item["previous_hash"]!=previous or item["hash"]!=expected:
            return False
        previous=item["hash"]
    return True

print(verify_chain(chain))


## 28 — Detect tampering

In [ ]:
tampered=copy.deepcopy(chain)
tampered[1]["record"]["decision"]="allow"
print("valid after modification:",verify_chain(tampered))


Hash chaining detects changes in this teaching example. Production audit integrity requires a broader design: protected writers, durable storage, signed/checkpointed roots, retention controls and independent access governance.

## 29 — Evidence bundle manifest

In [ ]:
manifest={
 "bundle_id":str(uuid.uuid4()),
 "window_start":(NOW-timedelta(hours=1)).isoformat(),
 "window_end":NOW.isoformat(),
 "records":len(chain),
 "chain_root":chain[-1]["hash"],
 "policy_versions":["v19","v20"],
 "schema_version":"1.0"
}
print(json.dumps(manifest,indent=2))


## 30 — Audit completeness

In [ ]:
protected_operations=10000
operations_with_decision_id=9994
coverage=operations_with_decision_id/protected_operations
print(f"audit coverage: {coverage:.3%}")


## 31 — Continuous control monitoring

In [ ]:
controls=pd.DataFrame([
 {"control":"R4 actions have decision evidence","passing":True},
 {"control":"Delegated writes have active delegation","passing":True},
 {"control":"Critical tools enforce decision IDs","passing":False},
 {"control":"Policy version always present","passing":True},
 {"control":"Sensitive tokens are redacted","passing":True},
])
controls


## 32 — Incident reconstruction

In [ ]:
incident=[
 {"time":"14:00:00","type":"authentication","id":"auth:1"},
 {"time":"14:00:03","type":"delegation","id":"del:483"},
 {"time":"14:00:05","type":"task","id":"task:483"},
 {"time":"14:00:06","type":"workload","id":"spiffe:claims"},
 {"time":"14:00:07","type":"authorization","id":"decision:9"},
 {"time":"14:00:08","type":"tool_call","id":"tool:77"},
 {"time":"14:00:09","type":"resource_change","id":"claim:483:v8"},
]
pd.DataFrame(incident)


## 33 — Causal reconstruction beats timestamps alone

In [ ]:
causal_edges=[
 ("auth:1","del:483"),
 ("del:483","task:483"),
 ("task:483","decision:9"),
 ("spiffe:claims","decision:9"),
 ("decision:9","tool:77"),
 ("tool:77","claim:483:v8"),
]
CG=nx.DiGraph(causal_edges)
print(list(nx.topological_sort(CG)))


## 34 — Retry awareness

In [ ]:
attempts=pd.DataFrame([
 {"request_id":"r1","attempt":1,"decision_id":"d1","outcome":"deny"},
 {"request_id":"r1","attempt":2,"decision_id":"d2","outcome":"allow"},
])
attempts


## 35 — Evidence quality scorecard

In [ ]:
quality={
 "completeness":0.9994,
 "integrity":1.0,
 "timeliness":0.98,
 "correlation":0.995,
 "provenance":1.0,
 "privacy":0.99,
 "reproducibility":0.97
}
quality


## 36 — Governance feedback

In [ ]:
recommendations=[
 "Investigate missing PEP decision IDs",
 "Review newly observed claim.export permission",
 "Compare v19 vs v20 critical allow rates",
 "Reduce unnecessary raw-content telemetry",
]
for r in recommendations:
    print("-",r)


## 37 — Adversarial regression tests

In [ ]:
# A denied action must never execute.
assert not joined[(joined.outcome=="deny") & (joined.executed==True)].empty, \
       "Lab intentionally contains a violation to detect"

# Detect a tool call with no decision.
assert tool_calls.decision_id.isna().any()

# Transaction parameter swap must change digest.
assert approved != changed

# Tampering must break evidence chain.
assert not verify_chain(tampered)

# Default deny and evaluation error must not collapse to same reason.
assert cedar_reason(cedar_default_deny) != cedar_reason(cedar_error)

print("All expected security conditions/detections demonstrated.")


## 38 — Real OpenTelemetry lab

Instrument a small agent → authorization service → tool service flow.

Create spans for:

```text
agent task
authorization request
tool call
resource operation
```

Propagate trace context through every hop.

Export through an OpenTelemetry Collector.

Then verify that:

```text
decision_id links PDP and PEP
trace_id links the workflow
critical telemetry contains no secrets
high-risk authorization events survive trace sampling
```

Use current OpenTelemetry GenAI semantic conventions where they fit, but keep your security event schema versioned because the GenAI conventions continue to evolve.


## 39 — Real OPA decision-log lab

Enable OPA decision logs.

Inspect:

```text
decision_id
input
result
timestamp
bundle metadata
metrics
```

Then use:

```text
policies/opa/observability.rego
```

to redact sensitive fields.

Verify the exported decision event contains the expected `masked`/`erased` metadata and no raw secret.


## 40 — SIEM lab

Send normalized authorization events to your preferred SIEM/log analytics platform.

Implement detections for:

```text
PEP bypass
critical allow from new workload
deny spike after policy deployment
cross-tenant attempt
delegation depth anomaly
quarantined agent authorization
```

Measure false positives and document response playbooks.


## 41 — Review questions

1. How is observability different from auditability?
2. Why are logs, traces and metrics not interchangeable?
3. What belongs in a canonical authorization event?
4. Why separate user, logical agent and workload identity?
5. What is delegation provenance?
6. Why capture task authority?
7. Why record policy/model/data versions?
8. Why are reason codes preferable to free-form text?
9. Why is an authorization explanation not chain-of-thought?
10. What does OPA decision logging provide?
11. How does OPA decision-log masking help?
12. What can Cedar diagnostics tell you?
13. Why distinguish default deny from evaluation error?
14. What should be logged for a ReBAC/OpenFGA check?
15. How does OpenTelemetry help authorization observability?
16. Why is raw GenAI content opt-in?
17. Which IDs should correlate an agent workflow?
18. Why avoid user/resource IDs as metric labels?
19. Why monitor suspicious allows?
20. What is policy drift analytics?
21. How can telemetry detect PEP bypass?
22. Why doesn't a PDP allow prove enforcement?
23. What is the decision-to-action gap?
24. How can critical parameters be bound to authorization?
25. What does hash chaining prove?
26. What does it *not* prove?
27. Why pseudonymize identifiers?
28. Why should redaction happen early?
29. Why should audit evidence not depend on trace sampling?
30. How do you measure audit completeness?
31. What belongs in an evidence packet?
32. How do continuous controls improve governance?
33. Why can timestamps alone be misleading in distributed systems?
34. How should retries be represented?
35. How should observability feed least-privilege improvement?

# Next course

## Intermediate 11 — Adversarial Authorization Testing for Agents
